# Exploratory survival-label baselines — not a clinical predictor

Uses the existing processed ZIP when the original raw CSV is absent. Original dataset provenance needs owner confirmation. An explicit reproducible sample of at most 20,000 rows keeps this learning experiment practical. Post-treatment dates/durations, handwritten risk scores, IDs and treatment choices are excluded from the prediction features.

In [ ]:
from load_data import load_existing
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, average_precision_score
df=load_existing()
if len(df)>20000:
    _,df=train_test_split(df, test_size=20000, stratify=df["survived"], random_state=42)
features=["age","gender","cancer_stage","family_history","smoking_status","bmi","cholesterol_level","hypertension","asthma","cirrhosis","other_cancer"]
X=df[features]; y=df["survived"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
print("Experiment rows:",len(df),"positive proportion:",y.mean())

In [ ]:
numeric=X.select_dtypes(include="number").columns.tolist()
categorical=[c for c in features if c not in numeric]
preprocessor=ColumnTransformer([("numeric",Pipeline([("fill",SimpleImputer()),("scale",StandardScaler())]),numeric),
    ("categories",Pipeline([("fill",SimpleImputer(strategy="most_frequent")),("encode",OneHotEncoder(handle_unknown="ignore"))]),categorical)])
for name,estimator in [("majority baseline",DummyClassifier(strategy="most_frequent")),
    ("logistic regression",LogisticRegression(max_iter=500,class_weight="balanced")),
    ("random forest",RandomForestClassifier(n_estimators=50,max_depth=12,class_weight="balanced",random_state=42,n_jobs=2))]:
    model=Pipeline([("preprocess",preprocessor), ("model",estimator)])
    model.fit(X_train,y_train)
    prediction=model.predict(X_test)
    print(name); print(classification_report(y_test,prediction,zero_division=0))
    print("Average precision:",average_precision_score(y_test,model.predict_proba(X_test)[:,1]))

## Interpretation
Compare minority recall/F1 and average precision with the majority baseline, not accuracy alone. No clinical usefulness, causal treatment effect, external validation or generalization is established. Sampling, data provenance and feature-availability assumptions limit this exercise.